# 🤖 ASSIGNMENT NLP – 3: Build a Chatbot using Hugging Face Transformers

**Internship:** Data Science Internship – February 2026  
**Task:** Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face.

---

## 📌 Objective
Build a console-based chatbot that communicates with users in natural language using a pre-trained transformer model (DialoGPT). The chatbot dynamically generates responses and simulates a basic conversational AI assistant.

---

## 🗂️ Table of Contents
1. [Install & Import Libraries](#1)
2. [Load Pre-trained Model](#2)
3. [Response Generation Function](#3)
4. [Conversation Flow Manager](#4)
5. [Run the Chatbot](#5)
6. [Sample Interaction Output](#6)
7. [Pipeline Summary](#7)

---
## 1. Install & Import Libraries <a id='1'></a>

We install the required libraries:  
- **`transformers`** – Hugging Face library to load pre-trained models  
- **`torch`** – PyTorch backend for running the model

In [ ]:
# Install required libraries (run once in Colab/Jupyter)
!pip install transformers torch --quiet

In [ ]:
# ─────────────────────────────────────────────────────────────
# Import all necessary libraries
# ─────────────────────────────────────────────────────────────

import torch                                          # PyTorch: deep learning framework
from transformers import AutoModelForCausalLM         # Auto-loads a causal language model
from transformers import AutoTokenizer                # Auto-loads the matching tokenizer

print("✅ Libraries imported successfully!")
print(f"   → PyTorch version  : {torch.__version__}")

# Detect device: use GPU if available, else fall back to CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   → Running on device: {DEVICE.upper()}")

---
## 2. Load Pre-trained Model <a id='2'></a>

We use **`microsoft/DialoGPT-medium`** — a GPT-2-based model fine-tuned specifically for multi-turn conversational dialogue.

| Property | Detail |
|---|---|
| Model | DialoGPT-medium |
| Base Architecture | GPT-2 |
| Task | Conversational Response Generation |
| Source | Hugging Face Model Hub |
| Parameters | ~345M |

> 💡 **Why DialoGPT?** Unlike vanilla GPT-2, DialoGPT is fine-tuned on Reddit conversation threads, making it ideal for dialogue generation.

In [ ]:
# ─────────────────────────────────────────────────────────────
# MODEL LOADING
# Load the pre-trained DialoGPT-medium model and its tokenizer
# ─────────────────────────────────────────────────────────────

MODEL_NAME = "microsoft/DialoGPT-medium"   # Hugging Face model identifier

print(f"⏳ Loading model: '{MODEL_NAME}' ...")
print("   (This may take a moment on first run — model weights are being downloaded)\n")

# Load the tokenizer — converts text ↔ token IDs
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="left"     # Pad from the left for open-ended generation
)

# Load the causal language model (DialoGPT)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)   # Move model to GPU (if available) or CPU
model.eval()               # Set model to evaluation mode (disables dropout)

print("✅ Model and Tokenizer loaded successfully!")
print(f"   → Model : {MODEL_NAME}")
print(f"   → Device: {DEVICE.upper()}")

---
## 3. Response Generation Function <a id='3'></a>

This function handles the core NLP pipeline:

```
User Text → Tokenize → Encode with History → Model Inference → Decode → Response Text
```

Key generation parameters:
- **`max_new_tokens`** – Limits the length of the generated reply
- **`temperature`** – Controls creativity (lower = more focused, higher = more creative)
- **`top_p`** – Nucleus sampling: keeps top tokens whose probabilities sum to `p`
- **`repetition_penalty`** – Penalises repeated phrases for more natural output

In [ ]:
# ─────────────────────────────────────────────────────────────
# RESPONSE GENERATION FUNCTION
# Encodes user input + conversation history, runs model inference,
# and decodes the generated token IDs back to readable text.
# ─────────────────────────────────────────────────────────────

def generate_response(user_input: str,
                      chat_history_ids=None,
                      max_new_tokens: int = 100,
                      temperature: float = 0.75,
                      top_p: float = 0.92,
                      repetition_penalty: float = 1.3):
    """
    Generate a chatbot response for the given user input.

    Parameters
    ----------
    user_input        : str   – The latest message from the user.
    chat_history_ids  : Tensor or None – Token IDs of previous turns (context).
    max_new_tokens    : int   – Maximum tokens to generate in the reply.
    temperature       : float – Sampling temperature (creativity control).
    top_p             : float – Nucleus sampling probability threshold.
    repetition_penalty: float – Penalty for repeating the same tokens.

    Returns
    -------
    response_text     : str   – The chatbot's reply as plain text.
    new_history_ids   : Tensor – Updated conversation history token IDs.
    """

    # ── Step 1: Tokenize the new user input ──────────────────
    # Append the EOS (end-of-sequence) token so the model knows
    # where one turn ends and the next begins.
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"          # Return PyTorch tensor
    ).to(DEVICE)

    # ── Step 2: Append to conversation history ───────────────
    # If there is prior history, concatenate it with the new input
    # along the sequence dimension (dim=1) so the model has context.
    if chat_history_ids is not None:
        bot_input_ids = torch.cat(
            [chat_history_ids, new_input_ids], dim=-1
        )
    else:
        # First turn — no history yet
        bot_input_ids = new_input_ids

    # ── Step 3: Model inference (generate reply tokens) ──────
    with torch.no_grad():            # Disable gradient computation (faster)
        new_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens      = max_new_tokens,
            pad_token_id        = tokenizer.eos_token_id,  # Avoid padding warnings
            do_sample           = True,                    # Enable sampling (vs greedy)
            temperature         = temperature,
            top_p               = top_p,
            repetition_penalty  = repetition_penalty,
        )

    # ── Step 4: Decode only the newly generated tokens ───────
    # Slice off the input tokens (bot_input_ids length) so we
    # only decode the model's reply, not the entire prompt.
    reply_ids = new_history_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(
        reply_ids[0],
        skip_special_tokens=True     # Remove EOS / padding tokens from output
    ).strip()

    # Fallback: if the model produces an empty response
    if not response_text:
        response_text = "I'm not sure I understood that. Could you rephrase?"

    return response_text, new_history_ids


print("✅ Response generation function defined.")

---
## 4. Conversation Flow Manager <a id='4'></a>

This function wraps the entire chatbot loop:
- Greets the user on startup
- Continuously reads user input
- Calls `generate_response()` for each turn
- Maintains conversation history across turns
- Exits cleanly when the user types `exit` or `quit`

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONVERSATION FLOW MANAGER
# Manages the full chatbot interaction loop with history context.
# ─────────────────────────────────────────────────────────────

def run_chatbot():
    """
    Launch the interactive chatbot loop.

    The chatbot:
      - Greets the user upon startup.
      - Reads user input in a loop.
      - Generates and prints a reply for each input.
      - Maintains multi-turn conversation context.
      - Terminates when the user types 'exit' or 'quit'.
    """

    print("=" * 60)
    print("          🤖  AI CHATBOT  (DialoGPT-medium)")
    print("=" * 60)
    print("  Type your message and press Enter to chat.")
    print("  Type 'exit' or 'quit' to end the conversation.")
    print("=" * 60)

    # ── Initial greeting ─────────────────────────────────────
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Initialise conversation history as None (empty at start)
    chat_history_ids = None

    # ── Main conversation loop ────────────────────────────────
    while True:

        # Read user input from console
        user_input = input("You: ").strip()

        # ── Exit condition ────────────────────────────────────
        if user_input.lower() in ("exit", "quit"):
            print("\nChatbot: Thank you for chatting with me. Goodbye! 👋")
            print("=" * 60)
            break

        # ── Ignore blank input ────────────────────────────────
        if not user_input:
            print("Chatbot: Please type something so I can assist you!\n")
            continue

        # ── Generate model response ───────────────────────────
        response, chat_history_ids = generate_response(
            user_input     = user_input,
            chat_history_ids = chat_history_ids
        )

        # ── Display the chatbot reply ─────────────────────────
        print(f"Chatbot: {response}\n")


print("✅ Chatbot function defined. Run the next cell to start chatting!")

---
## 5. Run the Chatbot <a id='5'></a>

> ▶️ **Execute the cell below to launch the chatbot.**  
> Type your messages in the input box and press **Enter**.  
> Type `exit` or `quit` to end the session.

In [ ]:
# ─────────────────────────────────────────────────────────────
# RUN THE CHATBOT
# Execute this cell to start the interactive conversation loop.
# ─────────────────────────────────────────────────────────────

run_chatbot()

---
## 6. Sample Interaction Output <a id='6'></a>

The following cell demonstrates a **simulated conversation** to showcase the expected chatbot behaviour (no interactive input required — outputs are pre-captured for review).

In [ ]:
# ─────────────────────────────────────────────────────────────
# SIMULATED CONVERSATION DEMO
# Runs a scripted list of user inputs through the chatbot
# and prints all responses. Useful for notebook output / review.
# ─────────────────────────────────────────────────────────────

def demo_conversation(sample_inputs: list):
    """
    Run a scripted demo conversation and display chatbot responses.

    Parameters
    ----------
    sample_inputs : list of str – Pre-defined user messages to send.
    """

    print("=" * 60)
    print("        📋  DEMO CONVERSATION OUTPUT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history_ids = None   # Reset history for a fresh demo

    for user_msg in sample_inputs:
        print(f"User   : {user_msg}")

        # Exit condition inside demo
        if user_msg.lower() in ("exit", "quit"):
            print("Chatbot: Thank you for chatting with me. Goodbye! 👋")
            break

        # Generate response
        response, chat_history_ids = generate_response(
            user_input       = user_msg,
            chat_history_ids = chat_history_ids
        )
        print(f"Chatbot: {response}\n")

    print("=" * 60)


# ── Define sample user inputs for the demo ───────────────────
sample_conversation = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Tell me about machine learning.",
    "Thank you",
    "exit"
]

# ── Run the demo ──────────────────────────────────────────────
demo_conversation(sample_conversation)

---
## 7. Pipeline Summary <a id='7'></a>

```
┌─────────────────────────────────────────────────────────────┐
│              CHATBOT PIPELINE OVERVIEW                      │
│                                                             │
│  User Input                                                 │
│      │                                                      │
│      ▼                                                      │
│  Tokenizer  ──► Encode text to token IDs                    │
│      │                                                      │
│      ▼                                                      │
│  Concat with History  ──► Provide multi-turn context        │
│      │                                                      │
│      ▼                                                      │
│  DialoGPT-medium Model  ──► Causal language modelling       │
│      │                                                      │
│      ▼                                                      │
│  Decode Output Tokens  ──► Convert IDs back to text         │
│      │                                                      │
│      ▼                                                      │
│  Display Response  ──► Print to console                     │
│      │                                                      │
│      ▼                                                      │
│  Loop Until 'exit'/'quit'                                   │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts Covered

| Concept | Description |
|---|---|
| **Transformer Architecture** | GPT-2 based auto-regressive model |
| **Tokenization** | Converting text ↔ token IDs via `AutoTokenizer` |
| **Conversation History** | Concatenating past turns as context |
| **Sampling Parameters** | `temperature`, `top_p`, `repetition_penalty` |
| **Inference Mode** | `torch.no_grad()` for memory-efficient generation |
| **Exit Handling** | Graceful loop termination on `exit`/`quit` |

---
*Notebook prepared as part of Data Science Internship – February 2026*